In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parents[1]
DATA_PROCESSED = ROOT / "data_processed"

in_path = DATA_PROCESSED / "04_final_filtered_dataset.parquet"
df = pd.read_parquet(in_path)


# -----------------------------
# Config: adjust column names if needed
# -----------------------------
SET_COL = "SetNo"                 # which set the point belongs to
GAME_NO_COL = "GameNo"            # game index within the set; tiebreak often recorded as 13
MATCH_COL = "match_id"


P1_GAMES_IN_SET_COL = "P1GamesWon"
P2_GAMES_IN_SET_COL = "P2GamesWon"

# Optional if you already created it; otherwise we fall back to (GameNo == 13)
TIEBREAK_COL = "tiebreak"         # binary flag (1 if tiebreak point)

# Optional metadata (nice to have in summary)
META_COLS = [c for c in ["year", "slam", "player1", "player2"] if c in df.columns]

# -----------------------------
# 1) Keep only matches that had a 3rd set played
# -----------------------------
matches_with_set3 = df.loc[df[SET_COL] == 3, MATCH_COL].unique()
df3 = df[df[MATCH_COL].isin(matches_with_set3)].copy()

# 3rd set only
set3 = df3[df3[SET_COL] == 3].copy()

# -----------------------------
# 2) Define tiebreak detection
# -----------------------------
if TIEBREAK_COL in set3.columns:
    set3["has_tiebreak_point"] = set3[TIEBREAK_COL].astype(bool)
else:
    set3["has_tiebreak_point"] = (set3[GAME_NO_COL] == 13)

# -----------------------------
# 3) "Close set" flags (reached 5–5 or 6–6 at any point)
# -----------------------------
# Safety: ensure required columns exist
missing = [c for c in [P1_GAMES_IN_SET_COL, P2_GAMES_IN_SET_COL] if c not in set3.columns]
if missing:
    raise KeyError(
        f"Missing required columns for games-in-set tracking: {missing}\n"
        "Rename P1_GAMES_IN_SET_COL / P2_GAMES_IN_SET_COL to your actual column names."
    )

set3["reached_55"] = (set3[P1_GAMES_IN_SET_COL] >= 5) & (set3[P2_GAMES_IN_SET_COL] >= 5)
set3["reached_66"] = (set3[P1_GAMES_IN_SET_COL] == 6) & (set3[P2_GAMES_IN_SET_COL] == 6)

# -----------------------------
# 4) Build a match-level summary for the 3rd set
# -----------------------------
# Sort so "last row" per match is meaningful
sort_cols = [MATCH_COL]
for c in [SET_COL, GAME_NO_COL]:
    if c in set3.columns:
        sort_cols.append(c)

set3 = set3.sort_values(sort_cols)

summary = (
    set3.groupby(MATCH_COL)
        .agg(
            **{c: (c, "first") for c in META_COLS},
            final_p1_games=(P1_GAMES_IN_SET_COL, "last"),
            final_p2_games=(P2_GAMES_IN_SET_COL, "last"),
            max_p1_games=(P1_GAMES_IN_SET_COL, "max"),
            max_p2_games=(P2_GAMES_IN_SET_COL, "max"),
            reached_5_5=("reached_55", "any"),
            reached_6_6=("reached_66", "any"),
            has_tiebreak=("has_tiebreak_point", "any"),
            max_game_no=(GAME_NO_COL, "max") if GAME_NO_COL in set3.columns else ("has_tiebreak_point", "size"),
        )
        .reset_index()
)

# A simple "close set" definition: reached 5-5 OR reached 6-6 OR has a tiebreak
summary["is_close_set3"] = summary["reached_5_5"] | summary["reached_6_6"] | summary["has_tiebreak"]

# -----------------------------
# 5) Filter what you want to inspect
# -----------------------------
close_set3_matches = summary[summary["is_close_set3"]].copy()

# Optional: spot potential "no-tiebreak final set" behavior
# e.g., sets going beyond 7 games for a player suggest play-until-2 or unusual encoding
close_set3_matches["suspect_long_set"] = (close_set3_matches["final_p1_games"] >= 8) | (close_set3_matches["final_p2_games"] >= 8)

print("Total matches with a 3rd set:", len(matches_with_set3))
print("3rd sets flagged as close:", len(close_set3_matches))
print("Close 3rd sets with potential long final set (>=8 games for someone):", close_set3_matches["suspect_long_set"].sum())

# Show the most relevant columns
cols_to_show = [MATCH_COL] + META_COLS + [
    "final_p1_games", "final_p2_games",
    "reached_5_5", "reached_6_6", "has_tiebreak",
    "suspect_long_set"
]
display(close_set3_matches[cols_to_show].sort_values(["suspect_long_set", "reached_6_6", "has_tiebreak"], ascending=False))


Total matches with a 3rd set: 91
3rd sets flagged as close: 14
Close 3rd sets with potential long final set (>=8 games for someone): 1


,match_id,year,slam,player1,player2,final_p1_games,final_p2_games,reached_5_5,reached_6_6,has_tiebreak,suspect_long_set
28,2013-wimbledon-2601,2013,wimbledon,Sabine Lisicki,Agnieszka Radwanska,9,7,True,True,True,True
13,2012-usopen-2501,2012,usopen,Victoria Azarenka,Samantha Stosur,7,6,True,True,True,False
58,2017-usopen-2503,2017,usopen,Venus Williams,Petra Kvitova,7,6,True,True,True,False
59,2017-usopen-2504,2017,usopen,Sloane Stephens,Anastasija Sevastova,7,6,True,True,True,False
70,2021-usopen-2503,2021,usopen,Elina Svitolina,Leylah Fernandez,6,7,True,True,True,False
80,2023-usopen-2602,2023,usopen,Madison Keys,Aryna Sabalenka,6,7,True,True,True,False
89,2024-wimbledon-2602,2024,wimbledon,Donna Vekic,Jasmine Paolini,6,7,True,True,True,False
3,2011-frenchopen-2502,2011,frenchopen,Anastasia Pavlyuchenkova,Francesca Schiavone,5,7,True,False,False,False
16,2012-usopen-2701,2012,usopen,Victoria Azarenka,Serena Williams,5,7,True,False,False,False
17,2012-wimbledon-2501,2012,wimbledon,Sabine Lisicki,Angelique Kerber,5,7,True,False,False,False
